In [76]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn .linear_model import LogisticRegression # (we can also use SVM)
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Activation
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix

In [54]:
df=pd.read_csv("spam_email_dataset.csv")
df

,email_id,subject,email_text,num_words,num_characters,num_exclamation_marks,num_links,has_suspicious_link,num_attachments,has_attachment,sender_email,sender_domain,sender_reputation_score,email_hour,email_day_of_week,is_weekend,num_recipients,contains_money_terms,contains_urgency_terms,label
0,0,Weekly Report,budget review - Statement our I claim world st...,19,114,0,2,0,2,1,lctvdzm@outlook.com,outlook.com,0.66,19,3,0,23,0,0,0
1,1,Project Update,team sync - President series today already. In...,18,114,0,7,0,0,0,pxyldmi@company.com,company.com,0.95,4,4,0,16,1,0,0
2,2,🔥WIN BIG NOW!!,win free urgent offer limited limited urgent u...,19,126,0,4,1,1,1,atvanls@unknownmail.cc,unknownmail.cc,0.68,3,0,0,10,1,1,1
3,3,🔥WIN BIG NOW!!,guarantee click now cash offer click now guara...,16,101,0,7,1,1,1,qalxcnf@chealdealz.xyz,chealdealz.xyz,0.69,19,5,1,25,1,1,1
4,4,Meeting Reminder,team sync - Significant property hotel not add...,18,111,0,7,1,2,1,xoiccxl@yahoo.com,yahoo.com,0.67,4,5,1,8,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9995,🔥WIN BIG NOW!!,free win cash win guarantee urgent free win li...,22,136,0,5,1,0,0,fefsrcz@unknownmail.cc,unknownmail.cc,0.68,8,1,0,29,1,1,1
9996,9996,You are selected!!!,click now free guarantee free win click now wi...,26,158,0,2,0,0,0,yxyctux@chealdealz.xyz,chealdealz.xyz,0.76,2,3,0,29,0,1,1
9997,9997,You are selected!!!,offer cash win limited click now cash limited ...,21,124,0,3,1,1,1,tkvqnki@unknownmail.cc,unknownmail.cc,0.69,6,1,0,25,0,1,1
9998,9998,Meeting Reminder,budget review - Many result affect idea.,7,40,0,1,0,2,1,wmvnrbn@outlook.com,outlook.com,0.80,7,2,0,18,0,0,0


In [55]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   email_id                 10000 non-null  int64  
 1   subject                  10000 non-null  object 
 2   email_text               10000 non-null  object 
 3   num_words                10000 non-null  int64  
 4   num_characters           10000 non-null  int64  
 5   num_exclamation_marks    10000 non-null  int64  
 6   num_links                10000 non-null  int64  
 7   has_suspicious_link      10000 non-null  int64  
 8   num_attachments          10000 non-null  int64  
 9   has_attachment           10000 non-null  int64  
 10  sender_email             10000 non-null  object 
 11  sender_domain            10000 non-null  object 
 12  sender_reputation_score  10000 non-null  float64
 13  email_hour               10000 non-null  int64  
 14  email_day_of_week      

In [56]:
df.describe()

,email_id,num_words,num_characters,num_exclamation_marks,num_links,has_suspicious_link,num_attachments,has_attachment,sender_reputation_score,email_hour,email_day_of_week,is_weekend,num_recipients,contains_money_terms,contains_urgency_terms,label
count,10000.00000,10000.000000,10000.000000,10000.0,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000
mean,4999.50000,18.097000,115.072400,0.0,3.976400,0.466700,0.803600,0.599600,0.681972,11.589600,3.00580,0.287100,15.501100,0.398500,0.499300,0.39950
std,2886.89568,5.163764,33.781247,0.0,2.580447,0.498915,0.752252,0.490004,0.172825,6.897023,2.00309,0.452431,8.650577,0.489614,0.500025,0.48982
min,0.00000,5.000000,25.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.300000,0.000000,0.00000,0.000000,1.000000,0.000000,0.000000,0.00000
25%,2499.75000,14.000000,91.000000,0.0,2.000000,0.000000,0.000000,0.000000,0.560000,6.000000,1.00000,0.000000,8.000000,0.000000,0.000000,0.00000
50%,4999.50000,18.000000,115.500000,0.0,4.000000,0.000000,1.000000,1.000000,0.690000,12.000000,3.00000,0.000000,16.000000,0.000000,0.000000,0.00000
75%,7499.25000,22.000000,139.000000,0.0,6.000000,1.000000,1.000000,1.000000,0.810000,18.000000,5.00000,1.000000,23.000000,1.000000,1.000000,1.00000
max,9999.00000,33.000000,221.000000,0.0,8.000000,1.000000,2.000000,1.000000,1.000000,23.000000,6.00000,1.000000,30.000000,1.000000,1.000000,1.00000


In [57]:
df.columns

Index(['email_id', 'subject', 'email_text', 'num_words', 'num_characters',
       'num_exclamation_marks', 'num_links', 'has_suspicious_link',
       'num_attachments', 'has_attachment', 'sender_email', 'sender_domain',
       'sender_reputation_score', 'email_hour', 'email_day_of_week',
       'is_weekend', 'num_recipients', 'contains_money_terms',
       'contains_urgency_terms', 'label'],
      dtype='object')

In [58]:
df.shape

(10000, 20)

In [59]:
df.isnull().sum()

,0
email_id,0
subject,0
email_text,0
num_words,0
num_characters,0
num_exclamation_marks,0
num_links,0
has_suspicious_link,0
num_attachments,0
has_attachment,0


In [60]:
df.dtypes

,0
email_id,int64
subject,object
email_text,object
num_words,int64
num_characters,int64
num_exclamation_marks,int64
num_links,int64
has_suspicious_link,int64
num_attachments,int64
has_attachment,int64


In [61]:
df.head()

,email_id,subject,email_text,num_words,num_characters,num_exclamation_marks,num_links,has_suspicious_link,num_attachments,has_attachment,sender_email,sender_domain,sender_reputation_score,email_hour,email_day_of_week,is_weekend,num_recipients,contains_money_terms,contains_urgency_terms,label
0,0,Weekly Report,budget review - Statement our I claim world st...,19,114,0,2,0,2,1,lctvdzm@outlook.com,outlook.com,0.66,19,3,0,23,0,0,0
1,1,Project Update,team sync - President series today already. In...,18,114,0,7,0,0,0,pxyldmi@company.com,company.com,0.95,4,4,0,16,1,0,0
2,2,🔥WIN BIG NOW!!,win free urgent offer limited limited urgent u...,19,126,0,4,1,1,1,atvanls@unknownmail.cc,unknownmail.cc,0.68,3,0,0,10,1,1,1
3,3,🔥WIN BIG NOW!!,guarantee click now cash offer click now guara...,16,101,0,7,1,1,1,qalxcnf@chealdealz.xyz,chealdealz.xyz,0.69,19,5,1,25,1,1,1
4,4,Meeting Reminder,team sync - Significant property hotel not add...,18,111,0,7,1,2,1,xoiccxl@yahoo.com,yahoo.com,0.67,4,5,1,8,0,0,0


In [62]:
y=df['label']
x=df.drop(['label'], axis=1)

In [63]:
x=pd.get_dummies(x,columns=['subject','email_text','sender_email','sender_domain'], drop_first=True)

In [64]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [65]:
scaler=StandardScaler()

In [66]:
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

In [67]:
model=Sequential()
model.add(Dense(128,activation='relu',input_dim=x_train.shape[1]))
model.add(Dense(64,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [80]:
from tensorflow.keras.losses import BinaryCrossentropy
model.compile(optimizer='adam',loss=BinaryCrossentropy(),metrics=['accuracy'])

In [82]:
history=model.fit(x_train,y_train,epochs=50,batch_size=32,validation_split=0.2)

Epoch 1/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 1.0000 - loss: 7.6589e-06 - val_accuracy: 0.4175 - val_loss: 1.6092
Epoch 2/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 1.0000 - loss: 6.7765e-06 - val_accuracy: 0.4175 - val_loss: 1.6127
Epoch 3/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - accuracy: 1.0000 - loss: 6.0256e-06 - val_accuracy: 0.4175 - val_loss: 1.6162
Epoch 4/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 1.0000 - loss: 5.3808e-06 - val_accuracy: 0.4169 - val_loss: 1.6196
Epoch 5/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 1.0000 - loss: 4.8221e-06 - val_accuracy: 0.4169 - val_loss: 1.6230
Epoch 6/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 1.0000 - loss: 4.3355e-06 - val_accuracy: 0.4162 - val_loss: 1.6264
Epoch 7/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 1.0000 - loss: 3.9107e-06 - val_accuracy: 0.4162 - val_loss: 1.6297
Epoch 8/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 7s 26ms/step - accuracy: 1.00

In [87]:
y_pred=model.predict(x_test)
y_pred=(y_pred>0.5).astype(int)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


In [88]:
accuracy=accuracy_score(y_test,y_pred)
precision=precision_score(y_test,y_pred)
recall=recall_score(y_test,y_pred)
f1=f1_score(y_test,y_pred)

In [89]:
print("Accuracy:",accuracy)
print("Precision:",precision)
print("Recall:",recall)
print("F1 Score:",f1)

Accuracy: 0.7985
Precision: 1.0
Recall: 0.4956195244055069
F1 Score: 0.6627615062761506
